# Summarization & Faithfulness Scoring

Generates an abstractive summary from the extracted knowledge graph and scores its faithfulness against the source text.

In [1]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import numpy as np

summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def generate_summary(text: str, max_length: int = 200) -> str:
    result = summarizer(text, max_length=max_length, min_length=60, do_sample=False)
    return result[0]['summary_text']

def faithfulness_score(source: str, summary: str) -> float:
    """Compute cosine similarity between source and summary embeddings."""
    src_emb = embedding_model.encode(source, convert_to_tensor=True)
    sum_emb = embedding_model.encode(summary, convert_to_tensor=True)
    score = util.cos_sim(src_emb, sum_emb).item()
    return round(score * 100, 1)

In [2]:
source_text = """
In January 2025, a landmark ceasefire agreement was brokered between Israel and Hamas,
mediated by Qatar, Egypt, and the United States. The deal outlined a phased withdrawal
of Israeli forces from Gaza, the release of hostages held by Hamas, and the entry of
humanitarian aid convoys.
"""

summary = generate_summary(source_text)
score = faithfulness_score(source_text, summary)

print(f'Summary: {summary}')
print(f'\nFaithfulness Score: {score}%')

Summary: In January 2025, a ceasefire was brokered between Israel and Hamas by Qatar, Egypt, and the US. The agreement includes phased withdrawal, hostage release, and humanitarian aid entry.

Faithfulness Score: 92.3%
